In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

BASE = Path(r"/teamspace/studios/this_studio/QA")
TRAIN_PATH = BASE / "data/Mysterious_Affair_at_Styles_Train_Data.csv"
BOOK_A_PATH = BASE / "data/BookA_test.csv"
BOOK_B_PATH = BASE / "data/BookB_test.csv"
BOOK_A_OUT = BASE / "submissions/BookA.csv"
BOOK_B_OUT = BASE / "submissions/BookB.csv"

# Create directories if they don't exist
BOOK_A_OUT.parent.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH)
book_a_df = pd.read_csv(BOOK_A_PATH)
book_b_df = pd.read_csv(BOOK_B_PATH)

EMBED_MODEL = SentenceTransformer("BAAI/bge-large-en-v1.5")
CE_MODEL = CrossEncoder('./models/fine_tuned_cross_encoder')

def clean_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    return re.sub(r"\s+", " ", text).strip()

def split_sentences(text: str) -> list[str]:
    text = clean_text(text)
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [part.strip() for part in parts if part.strip()]

def head_snippet(text: str, n_sentences: int = 2, max_chars: int = 700) -> str:
    sentences = split_sentences(text)
    snippet = " ".join(sentences[:n_sentences]) if sentences else clean_text(text)
    return snippet[:max_chars]

def tail_snippet(text: str, n_sentences: int = 2, max_chars: int = 700) -> str:
    sentences = split_sentences(text)
    snippet = " ".join(sentences[-n_sentences:]) if sentences else clean_text(text)
    return snippet[-max_chars:]

def chapter_start_flag(text: str) -> int:
    return int(bool(re.match(r"(?i)^\s*(chapter|part)\b", clean_text(text))))

def encode_texts(texts: list[str]) -> np.ndarray:
    return EMBED_MODEL.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

def build_representations(frame: pd.DataFrame) -> dict[str, object]:
    texts = [clean_text(t) for t in frame["text"].tolist()]
    heads = [head_snippet(t) for t in texts]
    tails = [tail_snippet(t) for t in texts]

    rep = {
        "texts": texts,
        "heads": heads,
        "tails": tails,
        "len": np.array([len(t) for t in texts], dtype=float),
        "chapter": np.array([chapter_start_flag(t) for t in texts], dtype=float),
        "full_emb": encode_texts(texts),
        "head_emb": encode_texts(heads),
        "tail_emb": encode_texts(tails),
    }
    return rep

def page_feature_matrix(rep: dict[str, object]) -> np.ndarray:
    lengths = rep["len"].reshape(-1, 1)
    chapters = rep["chapter"].reshape(-1, 1)
    return np.hstack(
        [
            rep["full_emb"],
            rep["head_emb"],
            rep["tail_emb"],
            lengths / max(float(lengths.max()), 1.0),
            chapters,
        ]
    )

print("Building representations (this may take a moment)...")
train_rep = build_representations(train_df)
book_a_rep = build_representations(book_a_df)
book_b_rep = build_representations(book_b_df)

all_features = np.vstack(
    [
        page_feature_matrix(train_rep),
        page_feature_matrix(book_a_rep),
        page_feature_matrix(book_b_rep),
    ]
)
scaler = StandardScaler()
scaled_all = scaler.fit_transform(all_features)

pca_dim = min(32, scaled_all.shape[1], scaled_all.shape[0] - 1)
pca = PCA(n_components=pca_dim, random_state=42)
pca.fit(scaled_all)

train_features = pca.transform(scaler.transform(page_feature_matrix(train_rep)))
book_a_features = pca.transform(scaler.transform(page_feature_matrix(book_a_rep)))
book_b_features = pca.transform(scaler.transform(page_feature_matrix(book_b_rep)))

# Train Global Absolute Position Model
train_target = np.linspace(0.0, 1.0, len(train_df), dtype=float)
position_model = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0])
position_model.fit(train_features, train_target)

# ==========================================
# HYBRID ARCHITECTURE LOGIC
# ==========================================

def is_mid_sentence_end(text: str) -> bool:
    if not text: return False
    # Captures missing punctuation OR ending in a comma/dash
    return bool(re.search(r'([^.!?\"\'”’]|[,—\-\^])\s*$', text.strip()))

def is_mid_sentence_start(text: str) -> bool:
    if not text: return False
    # Captures starting with lowercase OR starting with a quote mark then lowercase
    return bool(re.match(r'^[\'”’"]?\s*[a-z]', text.strip()))

def is_split_dialogue_end(text: str) -> bool:
    if not text: return False
    # Captures ending in 'said,' or 'asked,' with an optional opening quote
    return bool(re.search(r'[,:]\s*[\'”’"]?$', text.strip()))

def is_split_dialogue_start(text: str) -> bool:
    if not text: return False
    # Captures starting with a quote mark followed by a Capital letter
    return bool(re.match(r'^[\'”’"]\s*[A-Z]', text.strip()))

def build_hybrid_cost_matrix(rep: dict[str, object], pos_scores: np.ndarray, lambda_local=1.0, lambda_global=4.0) -> np.ndarray:
    N = len(pos_scores)
    tails = rep["tails"]
    heads = rep["heads"]
    
    # 1. Local Cost: Cosine Distance
    sim_matrix = np.zeros((N, N))
    for i in range(N):
        pairs = [[tails[i], heads[j]] for j in range(N)]
        sim_matrix[i] = CE_MODEL.predict(pairs)
        
    # Convert CE probabilities (0 to 1) into a cost (lower is better)
    local_cost = 1.0 - sim_matrix
    
    # 2. Global Cost: Flow Penalty
    delta = 1.0 / N
    global_cost = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            expected_j = pos_scores[i] + delta
            diff = abs(pos_scores[j] - expected_j)
            # Heavy penalty if the regression says j comes way before i (going backwards)
            if pos_scores[j] < pos_scores[i]:
                diff += 0.5 
            global_cost[i, j] = diff
            
    # Combine Costs
    cost_matrix = (lambda_local * local_cost) + (lambda_global * global_cost)
    np.fill_diagonal(cost_matrix, 9999.0) # Prevent self-loops
    
    # 3. Hard Links (Exact String Match Boost)
    tails = rep["tails"]
    heads = rep["heads"]
    for i in range(N):
        for j in range(N):
            if i != j:
                if is_mid_sentence_end(tails[i]) and is_mid_sentence_start(heads[j]):
                    if sim_matrix[i, j] > 0.4: # Must also have decent semantic overlap
                        cost_matrix[i, j] -= 3.0 # Huge discount to force this transition

                elif is_split_dialogue_end(tails[i]) and is_split_dialogue_start(heads[j]):
                    if sim_matrix[i, j] > 0.3: # slightly lower threshold for dialogue
                        cost_matrix[i, j] -= 2.0
                        
    # Ensure no negative costs for the TSP solver
    min_cost = np.min(cost_matrix)
    if min_cost < 0:
        cost_matrix -= min_cost
        
    return cost_matrix

def solve_tsp_path(cost_matrix: np.ndarray, time_limit_seconds: int = 5) -> list[int]:
    """Solves the Asymmetric TSP to find the optimal Hamiltonian path."""
    N = cost_matrix.shape[0]
    
    # Pad matrix with a dummy node to create an open path instead of a closed loop
    # Dummy node cost to/from any node is 0
    augmented_matrix = np.pad(cost_matrix, ((0, 1), (0, 1)), mode='constant', constant_values=0)
    
    # OR-Tools requires integer costs. Scale to preserve precision.
    int_matrix = (augmented_matrix * 10000).astype(int)

    # N+1 nodes, 1 vehicle, Start/End at dummy node (Index N)
    manager = pywrapcp.RoutingIndexManager(N + 1, 1, N)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(int_matrix[from_node][to_node])

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = time_limit_seconds

    solution = routing.SolveWithParameters(search_parameters)

    if not solution:
        print("Warning: TSP Solver failed to find a solution. Falling back to argsort.")
        return []

    # Extract sequence
    index = routing.Start(0)
    route = []
    while not routing.IsEnd(index):
        node_index = manager.IndexToNode(index)
        if node_index != N: # Exclude dummy node
            route.append(node_index)
        index = solution.Value(routing.NextVar(index))
    
    return route

def order_from_hybrid_tsp(rep: dict, features: np.ndarray, l_local=1.0, l_global=4.0) -> list[int]:
    # 1. Get absolute position guesses
    pos_scores = position_model.predict(features)
    
    # 2. Build Hybrid Cost Matrix
    cost_matrix = build_hybrid_cost_matrix(rep, pos_scores, lambda_local=l_local, lambda_global=l_global)
    
    # 3. Solve TSP
    order = solve_tsp_path(cost_matrix)
    
    # Fallback to pure regression if TSP fails
    if not order:
        order = list(np.argsort(pos_scores, kind="mergesort"))
        
    return order

# ==========================================
# EXECUTION & SAVING
# ==========================================

def save_submission(frame: pd.DataFrame, rep: dict, features: np.ndarray, out_path: Path, l_local=1.0, l_global=1.0) -> list[int]:
    order = order_from_hybrid_tsp(rep, features, l_local, l_global)
    submission = pd.DataFrame(
        {
            "original_page": np.arange(1, len(order) + 1, dtype=int),
            "shuffled_page": frame.iloc[order]["page"].to_numpy(dtype=int),
        }
    )
    submission.to_csv(out_path, index=False)
    return order

def kendall_on_self_shuffle(frame: pd.DataFrame, seed: int = 123, l_local=1.0, l_global=4.0) -> float:
    shuffled = frame.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    shuffled_rep = build_representations(shuffled)
    shuffled_features = pca.transform(scaler.transform(page_feature_matrix(shuffled_rep)))
    
    order = order_from_hybrid_tsp(shuffled_rep, shuffled_features, l_local, l_global)
    
    predicted_positions = np.empty(len(shuffled), dtype=int)
    for rank, idx in enumerate(order, start=1):
        predicted_positions[idx] = rank
    true_positions = shuffled["page"].to_numpy(dtype=int)
    tau = kendalltau(true_positions, predicted_positions).correlation
    return float(tau)

print("Tuning Hyperparameters on Training Data (This will take a minute)...")
best_tau = 0
best_weights = (1.0, 4.0)
for l_global in [1.0, 2.0, 4.0, 6.0]:
    for l_local in [1.0, 2.0, 4.0]:
        tau = kendall_on_self_shuffle(train_df, l_local=l_local, l_global=l_global)
        print(f"   Testing  Local: {l_local}, Global: {l_global}  Tau: {tau:.4f}")
        
        if tau > best_tau:
            best_tau = tau
            best_weights = (l_local, l_global)

print(f"\nOptimal Weights found: Local={best_weights[0]}, Global={best_weights[1]}")
print(f"Best Training Score: {best_tau:.4f}\n")

# Now use those best weights to solve the actual tests!
optimal_local = best_weights[0]
optimal_global = best_weights[1]



print("Solving Book A...")
book_a_order = save_submission(book_a_df, book_a_rep, book_a_features, BOOK_A_OUT, l_local=optimal_local, l_global=optimal_global)

print("Solving Book B...")
book_b_order = save_submission(book_b_df, book_b_rep, book_b_features, BOOK_B_OUT, l_local=optimal_local, l_global=optimal_global)

print("Running local validation on Training Data...")
train_tau = kendall_on_self_shuffle(train_df)
print(f"Train self-shuffle Kendall tau: {train_tau:.4f}")
print(f"BookA output rows: {len(book_a_order)} -> {BOOK_A_OUT.name}")
print(f"BookB output rows: {len(book_b_order)} -> {BOOK_B_OUT.name}")
print("Done.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Building representations (this may take a moment)...


Tuning Hyperparameters on Training Data (This will take a minute)...
   Testing  Local: 1.0, Global: 1.0  Tau: 0.3828
   Testing  Local: 2.0, Global: 1.0  Tau: 0.4135
   Testing  Local: 4.0, Global: 1.0  Tau: 0.3671
   Testing  Local: 1.0, Global: 2.0  Tau: 0.4455
   Testing  Local: 2.0, Global: 2.0  Tau: 0.4381
   Testing  Local: 4.0, Global: 2.0  Tau: 0.3934
   Testing  Local: 1.0, Global: 4.0  Tau: 0.3699
   Testing  Local: 2.0, Global: 4.0  Tau: 0.4635
   Testing  Local: 4.0, Global: 4.0  Tau: 0.3559
   Testing  Local: 1.0, Global: 6.0  Tau: 0.3965
   Testing  Local: 2.0, Global: 6.0  Tau: 0.4355
   Testing  Local: 4.0, Global: 6.0  Tau: 0.4065

Optimal Weights found: Local=2.0, Global=4.0
Best Training Score: 0.4635

Solving Book A...
Solving Book B...
Running local validation on Training Data...
Train self-shuffle Kendall tau: 0.3699
BookA output rows: 147 -> BookA.csv
BookB output rows: 57 -> BookB.csv
Done.
